# sentence_embeddings 測試

In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

# Mean Pooling - Take attention mask into account for correct averaging
# 计算文本序列的平均池化（mean pooling）嵌入，同时考虑到注意力掩码（attention mask），
# 以确保只对实际的输入标记（tokens）进行平均，而忽略填充标记（padding tokens）
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]  # First element of model_output contains all token embeddings
    # attention_mask 是一个掩码，用于指示输入中哪些 token 是有效的（通常为 1），哪些是填充标记（通常为 0）
    # input_mask_expanded 将 attention_mask 扩展为与 token_embeddings 形状相同，以便对每个 token 的嵌入都进行掩码
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    # 计算平均嵌入时，将 token_embeddings 按 attention_mask 加权求和（忽略填充值的嵌入），
    # 再除以有效 token 的总和（.sum(1))，即实际存在的标记数，dim=1代表求column sum，1e-9是為了避免除以0的錯誤
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# Sentences we want sentence embeddings for
sentences = ['This is an Apple', 'Apple']

# Load model from HuggingFace Hub
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

# Tokenize sentences
encoded_input = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')

# Compute token embeddings
with torch.no_grad():
    model_output = model(**encoded_input)

# Perform pooling
sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])

# Normalize embeddings
sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

# Compute cosine similarity between the two sentence embeddings
cosine_similarity = F.cosine_similarity(sentence_embeddings[0].unsqueeze(0), sentence_embeddings[1].unsqueeze(0))

print(f"Cosine similarity: {cosine_similarity.item()}")

Cosine similarity: 0.7473337054252625


In [2]:
# Sentence Embeddings的簡化版
from sentence_transformers import SentenceTransformer, util

# Sentences we want sentence embeddings for
sentences = ['This is an Apple', 'Apple']

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Sentence embeddings
sentence_embeddings = [model.encode(s) for s in sentences]

# Compute cosine similarity between the two sentence embeddings
cosine_similarity = util.pytorch_cos_sim(sentence_embeddings[0], sentence_embeddings[1])

print(f"Cosine similarity: {cosine_similarity.item()}")

Cosine similarity: 0.7473337650299072


# 讀資料集

In [3]:
import pandas as pd

df = pd.read_excel('training_data.xlsx')
#df = pd.read_excel('test_data.xlsx')
df.head()

,question_id,article,question,selection1,selection2,selection3,selection4,correct_answer
0,511,"In order to protect the environment, love the ...",What is the speed difference between Conscious...,50%,75%,100%,25%,1
1,6064,After graduating from Williamson University in...,What kind of work was library science thought ...,challenging,engaging,fulfilling,Routine work,4
2,3912,French scientists have developed a new technol...,What is the first method that the researchers ...,Ultrasound,Water flow,Laser,Airflow,4
3,3449,"In order to place apples on the shelves, fruit...",Which team recently proved that the pyramid sh...,A team of scientists from the University of Pa...,A French-German team,A team of researchers from the French National...,A team of fruit vendors,2
4,2302,The picture on the right is the Messier 9 star...,What type of celestial object is M9?,Galaxy,Nebula,Globular star cluster,Open star cluster,3


In [4]:
df.shape

(7046, 8)

In [5]:
selection = ['selection1','selection2','selection3','selection4']
selection.index('selection2')

1

# 統計標準答案是否會出現在文章中

In [6]:
import numpy as np
import pandas as pd

#df = pd.read_excel('training_data.xlsx')
df = pd.read_excel('test_data.xlsx')

selection = ['selection1','selection2','selection3','selection4']

# how many selections in text
count_dict = {0:0, 1:0, 2:0, 3:0, 4:0}

# how many selections in text and correct answer in the selections
corre_dict = {0:0, 1:0, 2:0, 3:0, 4:0}

count = 0

for i in range(df.shape[0]):
    rec = df.iloc[i]
    
    sel = 0
    cor = 0
    
    for s in selection:
        if rec[s].lower() in rec['article'].lower():
            if selection.index(s) + 1 == rec['correct_answer']:
                cor = 1 # s is correct answer and in text 
                
            sel = sel + 1
            
    count_dict[sel] = count_dict[sel] + 1
    corre_dict[sel] = corre_dict[sel] + cor
    
    if sel > 0:
        count = count + 1
            
print(count)

for i in count_dict:
    if i > 0:
        print(i, count_dict[i], '\t', np.round(count_dict[i]/count*100, 2), '%', '\t', np.round(corre_dict[i]/count*100, 2), '%')

1762
1 998 	 56.64 % 	 56.64 %
2 326 	 18.5 % 	 18.5 %
3 164 	 9.31 % 	 9.31 %
4 274 	 15.55 % 	 15.55 %


# 問答測試

In [7]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline, logging
logging.set_verbosity_error()

# 模型參考
#https://huggingface.co/models?pipeline_tag=question-answering&sort=trending
#https://huggingface.co/AIFS/Llama3-8x8B-Time-Agents

#model_name= "AIFS/Llama3-8x8B-Time-Agents"
model_name = "deepset/roberta-base-squad2"
#model_name = "google-bert/bert-large-uncased-whole-word-masking-finetuned-squad"

AG = pipeline('question-answering', model=model_name, tokenizer=model_name)
question = "Why is model conversion important?"
context = 'The option to convert models between FARM and transformers gives freedom to the user and let people easily switch between frameworks.'

# 获取答案
res = AG(question=question, context=context)

print(res['answer'])

gives freedom to the user


# 版本1 (直接預測答案，沒參考選項)
程式碼執行時間: 0:01:29.677330
共1135題正確；627題錯誤
Accuracy: 0.6441543700340522

In [8]:
from transformers import pipeline
from transformers import AutoTokenizer, AutoModel
import torch
import time
import torch.nn.functional as F
import numpy as np
import tqdm
from datetime import datetime as dt

# Define the mean pooling function
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]  # First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# Check if GPU is available and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load QA model
model_name = "deepset/roberta-base-squad2"
AG = pipeline('question-answering', model=model_name, tokenizer=model_name, device=0 if torch.cuda.is_available() else -1)

# Load embedding model
embed_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(embed_model_name)
model = AutoModel.from_pretrained(embed_model_name).to(device)

def answer_multiple_choice_question(text, question, answer):
    # Get the answer from the QA model
    result = AG(question=question, context=text)
    res_answer = result['answer']
    
    options = [answer, res_answer]  # 正確答案及預測結果塞入options
    
    encoded_input = tokenizer(options, padding=True, truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
        model_output = model(**encoded_input)
    sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
    sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

    cosine_sim = F.cosine_similarity(sentence_embeddings[0].unsqueeze(0), sentence_embeddings[1].unsqueeze(0))
    
    if cosine_sim.item() > 0.8:  # 正確答案與預測結果的相似度大於0.8時算正確
        return 1
    else:
        return 0

# Begin Predictions
ACC = 0

start_time = dt.now()

for i in tqdm.tqdm(range(len(df))):  # Use the full range len(df) in actual use
    try:
        now = df.iloc[i]
        
        text = str(now['article'])
        question = str(now['question'])
        options = [
                str(now["selection1"]),
                str(now["selection2"]),
                str(now["selection3"]),
                str(now["selection4"])]
        answer = options[int(now["correct_answer"])-1]
        
        if answer_multiple_choice_question(text, question, answer):
            ACC += 1
            
        if (i+1) % 500 == 0:
            tmp_time = dt.now()
            exe_time = tmp_time - start_time
            print(f"程式碼執行時間: {exe_time}")
            print(f"[{str(dt.now()):.19}] 跑完{i+1}筆資料")
            print(f"目前準確率{ACC}題正確，{i+1-ACC}題錯誤 準確率{ACC/(i+1)}\n")
    except:
        tmp_time = dt.now()
        exe_time = tmp_time - start_time
        print(f"程式碼執行時間: {exe_time}")
        print(f"[{str(dt.now()):.19}] 跑完{i}筆資料")
        print(f"共{ACC}題正確；{i-ACC}題錯誤")
        print(f"Accuracy: {ACC/i}\n")
        time.sleep(5)

end_time = dt.now()
exe_time = end_time - start_time

print(f"程式碼執行時間: {exe_time}")
print(f"共{ACC}題正確；{len(df)-ACC}題錯誤")
print(f"Accuracy: {ACC/len(df)}")

Using device: cuda


 29%|██████████████████████▌                                                        | 504/1762 [00:26<00:57, 21.84it/s]

程式碼執行時間: 0:00:25.920073
[2025-08-31 15:40:00] 跑完500筆資料
目前準確率308題正確，192題錯誤 準確率0.616



 57%|████████████████████████████████████████████▍                                 | 1005/1762 [00:51<00:32, 23.54it/s]

程式碼執行時間: 0:00:50.823020
[2025-08-31 15:40:25] 跑完1000筆資料
目前準確率636題正確，364題錯誤 準確率0.636



 85%|██████████████████████████████████████████████████████████████████▌           | 1503/1762 [01:15<00:11, 21.61it/s]

程式碼執行時間: 0:01:15.533086
[2025-08-31 15:40:50] 跑完1500筆資料
目前準確率972題正確，528題錯誤 準確率0.648



100%|██████████████████████████████████████████████████████████████████████████████| 1762/1762 [01:29<00:00, 19.77it/s]

程式碼執行時間: 0:01:29.143402
共1135題正確；627題錯誤
Accuracy: 0.6441543700340522


# 版本2 (預測答案後，從4個選項中找一個跟預測答案最接近的答案當答案)
程式碼執行時間: 0:01:23.806548<br>
共1437題正確；325題錯誤<br>
Accuracy: 0.815550510783201

In [9]:
import pandas as pd
from transformers import pipeline
from transformers import AutoTokenizer, AutoModel
import torch
import time
import torch.nn.functional as F
import numpy as np
import tqdm
from datetime import datetime as dt

# Define the mean pooling function
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]  # First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# Check if GPU is available and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load QA model on the specified device
model_name = "deepset/roberta-base-squad2"
AG = pipeline('question-answering', model=model_name, tokenizer=model_name, device=0 if device.type == "cuda" else -1)

# Load embedding model and move it to GPU
embed_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(embed_model_name)
model = AutoModel.from_pretrained(embed_model_name).to(device)

def answer_multiple_choice_question(text, question, options, answer):
    # Get the answer from the QA model
    result = AG(question=question, context=text)
    res_answer = result['answer']
    options.append(res_answer)  # 答案塞入options

    encoded_input = tokenizer(options, padding=True, truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
        model_output = model(**encoded_input)
    sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
    sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

    similarities = []
    for i in range(len(options) - 1):
        cosine_sim = F.cosine_similarity(sentence_embeddings[i].unsqueeze(0), sentence_embeddings[-1].unsqueeze(0))
        similarities.append(cosine_sim.item())  # Convert tensor to Python float

    best_option_index = np.argmax(similarities)
    if best_option_index + 1 == answer:
        return 1
    else:
        return 0

# Load your Excel file with pandas
df = pd.read_excel('test_data.xlsx')

ACC = 0

start_time = dt.now()

for i in tqdm.tqdm(range(len(df))):  # Use the full range len(df) in actual use
    try:
        now = df.iloc[i]
        
        text = str(now['article'])
        question = str(now['question'])
        options = [
            str(now["selection1"]),
            str(now["selection2"]),
            str(now["selection3"]),
            str(now["selection4"])
        ]
        answer = int(now["correct_answer"])
        
        if answer_multiple_choice_question(text, question, options, answer):
            ACC += 1

        if (i+1) % 500 == 0:
            tmp_time = dt.now()
            exe_time = tmp_time - start_time
            print(f"程式碼執行時間: {exe_time}")
            print(f"[{str(dt.now()):.19}] 跑完{i+1}筆資料")
            print(f"目前準確率{ACC}題正確，{i+1-ACC}題錯誤 準確率{ACC/(i+1)}\n")
    except Exception as e:
        tmp_time = dt.now()
        exe_time = tmp_time - start_time
        print(f"程式碼執行時間: {exe_time}")
        print(f"[{str(dt.now()):.19}] 跑完{i}筆資料")
        print(f"共{ACC}題正確；{i - ACC}題錯誤")
        print(f"Accuracy: {ACC / i}\n")
        print(f"Error: {e}")
        time.sleep(5)

end_time = dt.now()
exe_time = end_time - start_time
print(f"程式碼執行時間: {exe_time}")
print(f"共{ACC}題正確；{len(df) - ACC}題錯誤")
print(f"Accuracy: {ACC / len(df)}")

Using device: cuda


 29%|██████████████████████▌                                                        | 503/1762 [00:27<00:57, 22.05it/s]

程式碼執行時間: 0:00:27.613145
[2025-08-31 15:47:19] 跑完500筆資料
目前準確率395題正確，105題錯誤 準確率0.79



 57%|████████████████████████████████████████████▍                                 | 1003/1762 [00:54<00:39, 19.11it/s]

程式碼執行時間: 0:00:54.368163
[2025-08-31 15:47:46] 跑完1000筆資料
目前準確率812題正確，188題錯誤 準確率0.812



 85%|██████████████████████████████████████████████████████████████████▍           | 1502/1762 [01:20<00:11, 22.34it/s]

程式碼執行時間: 0:01:20.228181
[2025-08-31 15:48:12] 跑完1500筆資料
目前準確率1224題正確，276題錯誤 準確率0.816



100%|██████████████████████████████████████████████████████████████████████████████| 1762/1762 [01:34<00:00, 18.57it/s]

程式碼執行時間: 0:01:34.865396
共1437題正確；325題錯誤
Accuracy: 0.815550510783201


# 版本3 (預測答案後，從4個選項中找一個跟預測答案最接近的答案當答案，但會排除選項不再article中的選項)
程式碼執行時間: 0:01:27.658340<br>
共1608題正確；154題錯誤<br>
Accuracy: 0.9125993189557321

In [10]:
import pandas as pd
import torch
from transformers import pipeline, AutoTokenizer, AutoModel
import torch.nn.functional as F
import numpy as np
import tqdm
from datetime import datetime as dt
#from datasets import Dataset

# Check if GPU is available and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load QA model on the specified device
model_name = "deepset/roberta-base-squad2"
AG = pipeline('question-answering', model=model_name, tokenizer=model_name, device=0 if device.type == "cuda" else -1)

# Load embedding model and move it to GPU
embed_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(embed_model_name)
model = AutoModel.from_pretrained(embed_model_name).to(device)

def answer_multiple_choice_question(text, question, options, answer):
    # Get the answer from the QA model
    result = AG(question=question, context=text)
    res_answer = result['answer']
    # print(f"Model Answer:{res_answer} Answer:{options[answer-1]}")

    options.append(res_answer)  # Add the model's answer to the options
    encoded_input = tokenizer(options, padding=True, truncation=True, return_tensors='pt').to('cuda' if torch.cuda.is_available() else 'cpu')
    with torch.no_grad():
        model_output = model(**encoded_input)
    sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
    sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

    similarities = []
    for i in range(len(options) - 1):
        if options[i].lower() in text.lower():
            cosine_sim = F.cosine_similarity(sentence_embeddings[i].unsqueeze(0), sentence_embeddings[-1].unsqueeze(0))
            cosine_sim = cosine_sim.item()  # Convert tensor to Python float
        else:
            cosine_sim = 0.0
        similarities.append(cosine_sim)

    best_option_index = np.argmax(similarities)
    if best_option_index + 1 == answer:
        return 1
    else:
        return 0

ACC = 0

start_time = dt.now()

for i in tqdm.tqdm(range(len(df))):  # Use the full range len(df) in actual use
    try:
        now = df.iloc[i]

        text = str(now['article'])
        question = str(now['question'])
        options = [
            str(now["selection1"]),
            str(now["selection2"]),
            str(now["selection3"]),
            str(now["selection4"])]
        answer = int(now["correct_answer"])

        if answer_multiple_choice_question(text, question, options, answer):
            ACC += 1

        if (i + 1) % 500 == 0:
            tmp_time = dt.now()
            exe_time = tmp_time - start_time
            print(f"\u7a0b\u5f0f\u78b0\u57f7\u884c\u6642\u9593: {exe_time}")
            print(f"[{str(dt.now()):.19}] \u8dd1\u5b8c{i + 1}\u7b46\u8cc7\u6599")
            print(f"\u76ee\u524d\u6e96\u78ba\u7387{ACC}\u984c\u6b63\u78ba\uff0c{i + 1 - ACC}\u984c\u932f\u8aa4 \u6e96\u78ba\u7387{ACC / (i + 1)}\n")
    except Exception as e:
        tmp_time = dt.now()
        exe_time = tmp_time - start_time
        print(f"\u7a0b\u5f0f\u78b0\u57f7\u884c\u6642\u9593: {exe_time}")
        print(f"[{str(dt.now()):.19}] \u8dd1\u5b8c{i}\u7b46\u8cc7\u6599")
        print(f"\u5171{ACC}\u984c\u6b63\u78ba\uff1b{i - ACC}\u984c\u932f\u8aa4")
        print(f"Accuracy: {ACC / i}\n")
        print(f"Error: {e}")
        time.sleep(5)

end_time = dt.now()
exe_time = end_time - start_time
print(f"程式碼執行時間: {exe_time}")
print(f"共{ACC}題正確；{len(df) - ACC}題錯誤")
print(f"Accuracy: {ACC / len(df)}")

Using device: cuda


 29%|██████████████████████▌                                                        | 504/1762 [00:26<00:52, 24.16it/s]

程式碰執行時間: 0:00:26.022809
[2025-08-31 15:53:12] 跑完500筆資料
目前準確率445題正確，55題錯誤 準確率0.89



 57%|████████████████████████████████████████████▍                                 | 1003/1762 [00:51<00:34, 21.77it/s]

程式碰執行時間: 0:00:51.240671
[2025-08-31 15:53:37] 跑完1000筆資料
目前準確率911題正確，89題錯誤 準確率0.911



 85%|██████████████████████████████████████████████████████████████████▌           | 1505/1762 [01:17<00:10, 23.42it/s]

程式碰執行時間: 0:01:17.427488
[2025-08-31 15:54:03] 跑完1500筆資料
目前準確率1368題正確，132題錯誤 準確率0.912



100%|██████████████████████████████████████████████████████████████████████████████| 1762/1762 [01:31<00:00, 19.36it/s]

程式碼執行時間: 0:01:31.036561
共1608題正確；154題錯誤
Accuracy: 0.9125993189557321


# 版本4 (預測答案後，從4個選項中找一個跟預測答案最接近的答案當答案，但會排除選項不再article中的選項，若只有一個候選選項，則直接預測該選項)
程式碼執行時間: 0:01:27.196759<br>
共1612題正確；150題錯誤<br>
Accuracy: 0.9148694665153235

In [12]:
# Load your Excel file with pandas
df = pd.read_excel('test_data.xlsx')

selection = ['selection1','selection2','selection3','selection4']

count_dict = {0:0, 1:0, 2:0, 3:0, 4:0}
n_sel = []

count = 0

for i in range(df.shape[0]):
    rec = df.iloc[i]
    
    sel = 0
    
    for s in selection:
        if rec[s].lower().strip() in rec['article'].lower().strip():
            sel = sel + 1
            
    count_dict[sel] = count_dict[sel] + 1
    n_sel.append(sel)
    
    if sel > 0:
        count = count + 1
            
print(count)

for i in count_dict:
    print(i, count_dict[i], '\t', np.round(count_dict[i]/count*100, 1), '%')
    
print(n_sel[:15])

1762
0 0 	 0.0 %
1 998 	 56.6 %
2 326 	 18.5 %
3 164 	 9.3 %
4 274 	 15.6 %
[1, 1, 2, 3, 2, 1, 4, 1, 4, 1, 1, 1, 1, 1, 1]


In [14]:
import pandas as pd
import torch
from transformers import pipeline, AutoTokenizer, AutoModel
import torch.nn.functional as F
import numpy as np
from datetime import datetime as dt
from datasets import Dataset, DatasetDict
from torch.utils.data import DataLoader
from tqdm import tqdm

# Define the mean pooling function
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]  # First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# Load QA model
model_name = "deepset/roberta-base-squad2"
#model_name = "google-bert/bert-large-uncased-whole-word-masking-finetuned-squad"
#model_name = "timpal0l/mdeberta-v3-base-squad2"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

AG = pipeline('question-answering', model=model_name, tokenizer=model_name, device=0 if torch.cuda.is_available() else -1)

# Load embedding model
embed_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(embed_model_name)
model = AutoModel.from_pretrained(embed_model_name).to(device)

def answer_multiple_choice_question(data):
    text = data['article'].strip()
    question = data['question']
    options = [
        data["selection1"].strip(),
        data["selection2"].strip(),
        data["selection3"].strip(),
        data["selection4"].strip()
    ]
    answer = data["correct_answer"]

    # Get the answer from the QA model
    result = AG(question=question, context=text)
    res_answer = result['answer']
    
    options.append(res_answer)  # Add the model's answer to the options
    encoded_input = tokenizer(options, padding=True, truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
        model_output = model(**encoded_input)
    sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
    sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

    similarities = []
    for i in range(len(options) - 1):
        if options[i].lower() in text.lower().strip():
            if data['n_sel'] == 1:
                cosine_sim = 1.0
            else:
                cosine_sim = F.cosine_similarity(sentence_embeddings[i].unsqueeze(0), sentence_embeddings[-1].unsqueeze(0))
                cosine_sim = cosine_sim.item() # Convert tensor to Python float
        else:
            cosine_sim = 0.0
            
        similarities.append(cosine_sim)

    best_option_index = np.argmax(similarities)
    
    if best_option_index + 1 == answer:
        return 1
    else:
        if data['n_sel'] == 1:
            print(data['selection1'], '/', data['selection2'], '/', data['selection3'], '/', data['selection4'], '/', data['correct_answer'], '-->', best_option_index + 1)
        else:
            return 0

# Load your Excel file with pandas
df = pd.read_excel('test_data.xlsx')
df['n_sel'] = n_sel

# Convert pandas DataFrame to Hugging Face dataset
dataset = Dataset.from_pandas(df)

# Use a DataLoader for efficient batching
data_loader = DataLoader(dataset, batch_size=8, shuffle=False)

ACC = 0
results = []

start_time = dt.now()

for batch in tqdm(data_loader):
    for i in range(len(batch['article'])):
        try:
            data = {key: batch[key][i] for key in batch}
            if answer_multiple_choice_question(data):
                ACC += 1
                results.append('Correct')
            else:
                results.append('Wrong')
        except Exception as e:
            print(f"Error: {e}")

end_time = dt.now()
exe_time = end_time - start_time
print(f"程式碼執行時間: {exe_time}")
print(f"共{ACC}題正確；{len(dataset) - ACC}題錯誤")
print(f"Accuracy: {ACC / len(dataset)}")

cuda


100%|████████████████████████████████████████████████████████████████████████████████| 221/221 [01:22<00:00,  2.66it/s]

程式碼執行時間: 0:01:22.969553
共1612題正確；150題錯誤
Accuracy: 0.9148694665153235


In [6]:
# Create the cross-tabulation
cross_tab = pd.crosstab(n_sel, results)
cross_tab['%'] = cross_tab['Wrong'] / (cross_tab['Correct'] + cross_tab['Wrong'])

print(cross_tab)

col_0  Correct  Wrong         %
row_0                          
1          998      0  0.000000
2          274     52  0.159509
3          127     37  0.225610
4          213     61  0.222628


# 版本5 (預測答案後，從4個選項中找一個跟預測答案最接近的答案當答案，但會排除選項不再article中的選項，若只有一個候選選項，則直接預測該選項，更換預測模型)
程式碼執行時間: 0:02:48.098942<br>
共1616題正確；146題錯誤<br>
Accuracy: 0.9171396140749148

In [15]:
import pandas as pd
import torch
from transformers import pipeline, AutoTokenizer, AutoModel
import torch.nn.functional as F
import numpy as np
from datetime import datetime as dt
from datasets import Dataset, DatasetDict
from torch.utils.data import DataLoader
import tqdm

# Define the mean pooling function
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]  # First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# Load QA model
#model_name = "deepset/roberta-base-squad2"
#model_name = "google-bert/bert-large-uncased-whole-word-masking-finetuned-squad"
model_name = "timpal0l/mdeberta-v3-base-squad2"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

AG = pipeline('question-answering', model=model_name, tokenizer=model_name, device=0 if torch.cuda.is_available() else -1)

# Load embedding model
embed_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(embed_model_name)
model = AutoModel.from_pretrained(embed_model_name).to(device)

def answer_multiple_choice_question(data):
    text = data['article'].strip()
    question = data['question']
    options = [
        data["selection1"].strip(),
        data["selection2"].strip(),
        data["selection3"].strip(),
        data["selection4"].strip()
    ]
    answer = data["correct_answer"]

    # Get the answer from the QA model
    result = AG(question=question, context=text)
    res_answer = result['answer']
    
    options.append(res_answer)  # Add the model's answer to the options
    encoded_input = tokenizer(options, padding=True, truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
        model_output = model(**encoded_input)
    sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
    sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

    similarities = []
    for i in range(len(options) - 1):
        if options[i].lower() in text.lower().strip():
            if data['n_sel'] == 1:
                cosine_sim = 1.0
            else:
                cosine_sim = F.cosine_similarity(sentence_embeddings[i].unsqueeze(0), sentence_embeddings[-1].unsqueeze(0))
                cosine_sim = cosine_sim.item() # Convert tensor to Python float
        else:
            cosine_sim = 0.0
            
        similarities.append(cosine_sim)

    best_option_index = np.argmax(similarities)
    
    if best_option_index + 1 == answer:
        return 1
    else:
        if data['n_sel'] == 1:
            print(data['selection1'], '/', data['selection2'], '/', data['selection3'], '/', data['selection4'], '/', data['correct_answer'], '-->', best_option_index + 1)
        else:
            return 0

# Load your Excel file with pandas
df = pd.read_excel('test_data.xlsx')
df['n_sel'] = n_sel

# Convert pandas DataFrame to Hugging Face dataset
dataset = Dataset.from_pandas(df)

# Use a DataLoader for efficient batching
data_loader = DataLoader(dataset, batch_size=8, shuffle=False)

ACC = 0
results = []

start_time = dt.now()

for batch in tqdm.tqdm(data_loader):
    for i in range(len(batch['article'])):
        try:
            data = {key: batch[key][i] for key in batch}
            if answer_multiple_choice_question(data):
                ACC += 1
                results.append('Correct')
            else:
                results.append('Wrong')
        except Exception as e:
            print(f"Error: {e}")

end_time = dt.now()
exe_time = end_time - start_time
print(f"程式碼執行時間: {exe_time}")
print(f"共{ACC}題正確；{len(dataset) - ACC}題錯誤")
print(f"Accuracy: {ACC / len(dataset)}")

cuda


100%|████████████████████████████████████████████████████████████████████████████████| 221/221 [03:06<00:00,  1.18it/s]

程式碼執行時間: 0:03:06.515291
共1616題正確；146題錯誤
Accuracy: 0.9171396140749148


# 版本6 (直接請T5-Flan-T5 Base 模型預測結果 (Zero-Shot))
Accuracy: 0.9188422247446084

In [16]:
import pandas as pd
pd.set_option('display.max_rows', None)

import warnings
#warnings.simplefilter("ignore")

train = pd.read_excel('training_data.xlsx')
test = pd.read_excel('test_data.xlsx')
dev = train.sample(n=1000, random_state=1)

test.head()

,question_id,article,question,selection1,selection2,selection3,selection4,correct_answer
0,4967,The Center for Advanced European Survey and Re...,Where is the sensor planned to be installed?,On the rim of the tire,Inside the tire,In the tread groove of the tire where it conta...,In the sidewall of the tire,3
1,3465,When organisms evolve from ocean to freshwater...,What type of organism were the fossils identif...,bacteria,eukaryotes,archaea,prokaryotes,2
2,7649,What is LCD? Liquid crystal is a material that...,What is the abbreviation of liquid crystal dis...,QLED,OLED,LED,LCD,4
3,3881,Physicians generally agree that sex improves m...,Which country's researchers mentioned in a rep...,Germany,United States,Japan,South Korea,1
4,3726,The government has recently begun to promote c...,What has been a hot topic in Taiwan for the la...,opening of health insurance and medical data,development of semiconductor,opening of the Forbidden City's cultural resou...,opening of high school exams,1


In [17]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(device)

# Load model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

try:
    # 加载模型
    tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
    model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").to(device)
    print("Model and tokenizer loaded successfully!")
except Exception as e:
    print(f"Error loading model or tokenizer: {e}")
    print("Please check your internet connection, available disk space, or Hugging Face cache.")

cuda:0
Model and tokenizer loaded successfully!


In [18]:
preamble = \
    'Use the articles about the question, '\
    'Answer the following question by outputting the letters A, B, C, D, '\
    'in order of the most likely to be correct to the to least likely to be correct.'
preamble

'Use the articles about the question, Answer the following question by outputting the letters A, B, C, D, in order of the most likely to be correct to the to least likely to be correct.'

In [19]:
from string import Template

preamble = \
    'Use the articles about the question, '\
    'Answer the following question by outputting the letters A, B, C, D, '\
    'in order of the most likely to be correct to the to least likely to be correct.'

template = Template('$preamble\n\n$article\n\nThe question is listed below.\n$question\n\nA) $a\nB) $b\nC) $c\nD) $d')

In [20]:
from string import Template

preamble = \
    'Using the following article '\
    'to Answer the question by outputting only one letter from A, B, C, or D '\
    'that you believe is the most likely to be correct.'

template = Template('$preamble\n\n$article\n\nThe question is listed below.\n$question\n\nA) $a\nB) $b\nC) $c\nD) $d')

In [21]:
def format_input(df, idx):   
    article = df.loc[idx, 'article']
    question = df.loc[idx, 'question']
    a = df.loc[idx, 'selection1']
    b = df.loc[idx, 'selection2']
    c = df.loc[idx, 'selection3']
    d = df.loc[idx, 'selection4']

    input_text = template.substitute(
        preamble=preamble,article=article, question=question, a=a, b=b, c=c, d=d)
    
    return input_text

print(format_input(test, 0))

Using the following article to Answer the question by outputting only one letter from A, B, C, or D that you believe is the most likely to be correct.

The Center for Advanced European Survey and Research (CAESAR) has developed a new measurement sensor that can alert drivers to road conditions. Although cars have been equipped with safety devices such as ABS systems for many years, they have functions such as preventing tire locking or slipping, properly distributing tire grip, and maintaining vehicle driving stability. However, these devices still have a major drawback, which is that the system responds too slowly, and the system does not display the warning until the brakes are applied or the vehicle is already spinning. The sensor designed by Kaiser Research Center has the ability to continuously measure tire grip, identify dangerous road conditions early, shorten braking time, and quickly restore the balance system. In order for the sensor to fully grasp the ground conditions, a la

In [25]:
inputs = tokenizer(format_input(train, 1), return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=1)
answer = tokenizer.batch_decode(outputs, skip_special_tokens=True)

print(answer)

['D']


In [22]:
import tqdm
from sklearn.metrics import classification_report, accuracy_score

### 对验证集进行输出
for idx in tqdm.tqdm(dev.index):
    inputs = tokenizer(format_input(dev, idx), return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_new_tokens=1)
    answer = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    dev.loc[idx, 'prediction'] = answer[0]
    
result = dev["prediction"].str[0].map({"A": 1, "B": 2, "C": 3, "D": 4}).fillna(1)
y_true = dev['correct_answer']
print(classification_report(y_true,result))
print(f"Accuracy: {accuracy_score(y_true,result)}")

100%|██████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:18<00:00, 12.76it/s]

              precision    recall  f1-score   support

           1       0.94      0.91      0.92       249
           2       0.91      0.94      0.92       270
           3       0.94      0.91      0.93       250
           4       0.89      0.92      0.91       231

    accuracy                           0.92      1000
   macro avg       0.92      0.92      0.92      1000
weighted avg       0.92      0.92      0.92      1000

Accuracy: 0.921


In [23]:
import tqdm
from sklearn.metrics import classification_report

# 对測試集进行输出
for idx in tqdm.tqdm(test.index):
    inputs = tokenizer(format_input(test, idx), return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_new_tokens=1)
    answer = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    test.loc[idx, 'Predicted_Results'] = answer[0]
    
result = test["Predicted_Results"].str[0].map({"A": 1, "B": 2, "C": 3, "D": 4}).fillna(1)
y_true = test['correct_answer']
print(classification_report(y_true,result))
print(f"Accuracy: {accuracy_score(y_true,result)}")

100%|██████████████████████████████████████████████████████████████████████████████| 1762/1762 [02:18<00:00, 12.72it/s]

              precision    recall  f1-score   support

           1       0.95      0.90      0.92       443
           2       0.93      0.93      0.93       430
           3       0.90      0.92      0.91       451
           4       0.91      0.94      0.93       438

    accuracy                           0.92      1762
   macro avg       0.92      0.92      0.92      1762
weighted avg       0.92      0.92      0.92      1762

Accuracy: 0.9216799091940976


In [35]:
test["Predicted_Results"] = test["Predicted_Results"].map({"A":1,"B":2,"C":3,"D":4})
test[['question_id', 'Predicted_Results']].to_csv('Final_Results.csv', index=False)